## 01_研究問題與計畫
本筆記本的目標有三個：

固定本專題的研究問題：明確定義自變數、依變數與統計假設。

描述變數特徵：定義欄位在統計學中的資料型態。

從原始資料表中擷取核心變數：輸出包含特定欄位的新資料表，方便後續分析。

### 1. Research Question (研究問題)
單因子變異數分析 (One-way ANOVA)

英文問題：Is there a significant difference in the average daily total screen time among students with different frequencies of physical fighting?

中文問題：在 YRBS 2007 資料中，參與不同肢體衝突（打架）頻率的高中生，其「每日平均總螢幕使用時間（電視 + 電玩）」是否存在顯著差異？

📊 專案變數與方法設計
| 項目內容 | 變數名稱 | 變數角色 | 統計資料型態 (Data Type) | 說明 |
| :--- | :--- | :--- | :--- | :--- |
| **分組變數** | `PhysicalFighting` | 自變數 (Independent Variable) | 類別變數 / 順序尺度 | 過去12個月打架次數（後續將分為3組） |
| **反應變數** | `TelevisionWatching` | 依變數的一部分 | 類別代號（需還原為小時） | 上學日平均每天看電視時數 |
| **反應變數** | `ComputerUse` | 依變數的一部分 | 類別代號（需還原為小時） | 平均每天玩電腦/打電玩時數 |
| **反應變數** | `Total_Screen_Time` | 最終依變數 (Dependent Variable) | **連續變數 / 等比尺度** | 電視時數 + 電玩時數的加總 |
| **分析方法** | **One-way ANOVA** | 統計檢定方法 | - | 比較三組或以上獨立樣本的平均數 |

### 2. Statistical Hypotheses (統計假設)
我們將根據統計學規範，設立虛無假設與對立假設。未來我們將把打架次數分為三組：組別1（0次）、組別2（1-3次）、組別3（4次以上）。

虛無假設 ($H_0$)：$\mu_1 = \mu_2 = \mu_3$（不同打架頻率的高中生，其每日平均總螢幕使用時間皆相同）

對立假設 ($H_1$)：$\mu_1, \mu_2, \mu_3$ 不全相等（至少有一組高中生的每日平均總螢幕使用時間與其他組別存在顯著差異 

In [4]:
# ==========================================
# 步驟 1：匯入套件與設定環境
# ==========================================
import pandas as pd
import os

# 設定顯示格式，確保資料顯示完整
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# ==========================================
# 步驟 2：讀取原始資料 (注意路徑)
# ==========================================
# 因為 01_Question.ipynb 在 notebook 資料夾內，所以要用 ../ 回到上一層找到 data
input_path = "../data/raw/YRBS_2007.csv"

print("正在讀取原始 CSV 資料...")
df = pd.read_csv(input_path)
print("原始資料維度：", df.shape)

# ==========================================
# 步驟 3：指定專題要使用的 3 個核心原始變數
# ==========================================
selected_columns = ["PhysicalFighting", "TelevisionWatching", "ComputerUse"]

# 擷取指定欄位並複製一張新表
df_selected = df[selected_columns].copy()

print("\n擷取後資料維度：", df_selected.shape)
print("\n前 5 列原始數據預覽：")
display(df_selected.head())

正在讀取原始 CSV 資料...
原始資料維度： (14041, 103)

擷取後資料維度： (14041, 3)

前 5 列原始數據預覽：


,PhysicalFighting,TelevisionWatching,ComputerUse
0,8.0,4.0,2.0
1,NaN,6.0,3.0
2,4.0,4.0,3.0
3,1.0,1.0,1.0
4,NaN,7.0,7.0


In [6]:
# ==========================================
# 步驟 4：清洗缺失值與數值還原
# ==========================================
print("正在剔除任一核心欄位中有缺失值(NaN)的學生資料...")
df_cleaned = df_selected.dropna().copy()
print(f"剔除完畢！有效學生樣本數：{df_cleaned.shape[0]} 筆。")

# 建立 CDC 2007 問卷代號還原成實際小時的對照表
screen_time_map = {
    1: 0.0,   # 不看 / 不玩
    2: 0.5,   # 不到 1 小時
    3: 1.0,   # 1 小時
    4: 2.0,   # 2 小時
    5: 3.0,   # 3 小時
    6: 4.0,   # 4 小時
    7: 5.0    # 5 小時以上
}

# 衍生計算實際小時數
df_cleaned['TV_Hours'] = df_cleaned['TelevisionWatching'].map(screen_time_map)
df_cleaned['Game_Hours'] = df_cleaned['ComputerUse'].map(screen_time_map)

# 計算最終反應變數 (Y值)：總螢幕使用時間
df_cleaned['Total_Screen_Time'] = df_cleaned['TV_Hours'] + df_cleaned['Game_Hours']

# 只保留最乾淨的 4 個欄位
final_df = df_cleaned[['PhysicalFighting', 'TelevisionWatching', 'ComputerUse', 'Total_Screen_Time']]

# ==========================================
# 步驟 5：輸出乾淨的 CSV 檔
# ==========================================
output_dir = "../data/processed"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

output_path = "../data/processed/YRBS_2007_cleaned.csv"
final_df.to_csv(output_path, index=False)

print(f"\n[成功] 乾淨的專題資料表已輸出至：{output_path}")
print("輸出資料表前 5 列確認：")
display(final_df.head())

正在剔除任一核心欄位中有缺失值(NaN)的學生資料...
剔除完畢！有效學生樣本數：13479 筆。

[成功] 乾淨的專題資料表已輸出至：../data/processed/YRBS_2007_cleaned.csv
輸出資料表前 5 列確認：


,PhysicalFighting,TelevisionWatching,ComputerUse,Total_Screen_Time
0,8.0,4.0,2.0,2.5
2,4.0,4.0,3.0,3.0
3,1.0,1.0,1.0,0.0
5,1.0,1.0,1.0,0.0
6,1.0,6.0,2.0,4.5


### 3. Output (預期輸出產物說明)
本筆記本最終輸出的檔案為：../data/processed/YRBS_2007_cleaned.csv

此檔案為本專題的核心資料庫，僅包含後續進行敘述統計與 ANOVA 檢定所需的 4 個關鍵項目：

1. PhysicalFighting：高中生的打架次數群組（自變數 $X$ 基底）。

2. TelevisionWatching：原始電視看時間代號。

3. ComputerUse：原始電腦電玩時間代號。

4. Total_Screen_Time：還原後相加的每日總螢幕使用小時數（反應變數 $Y$）。

後續我們將開闢 02_Descriptive_Statistics.ipynb，針對此檔案進行分組與平均數計算。